<h1 style="color:darkcyan;">Predicción del Ciclo Económico Español mediante Machine Learning</h1>

La economía española atraviesa periódicamente fases de expansión, desaceleración, recesión y recuperación. Anticipar en qué fase se encontrará la economía en los próximos meses tiene un valor enorme para empresas, inversores y responsables de política económica.

**El objetivo de este proyecto es construir un modelo de Machine Learning capaz de predecir la fase del ciclo económico español para los próximos 12 meses**, utilizando indicadores macroeconómicos publicados por organismos oficiales españoles y europeos.

La variable objetivo será la **fase del ciclo** (Expansión, Desaceleración, Recesión o Recuperación), determinada principalmente por la evolución del PIB español. Un modelo que acierte la fase con antelación suficiente permite, por ejemplo, anticipar si el desempleo va a subir, si el crédito se va a contraer o si la demanda interna va a debilitarse.

Abordaremos el problema de principio a fin:

* Entender el contexto económico español y las fuentes de datos disponibles
* Identificar y comprender las variables relevantes
* Obtener, limpiar y explorar los datos
* Crear nuevas variables predictoras (feature engineering)
* Entrenar y comparar varios modelos de clasificación
* Evaluar el rendimiento, diagnosticar errores e interpretar los resultados

---

**Especificaciones del trabajo:**

* Comenta todo el código para garantizar su legibilidad. Interpreta los resultados obtenidos en cada paso.
* El trabajo debe ser original. No se permite el uso de herramientas de inteligencia artificial generativa. Desarrolla directamente en Jupyter Notebook sobre Anaconda.
* Trabajo en grupos de dos personas (los mismos grupos del curso).
* No uses datos del futuro para predecir el pasado (data leakage). En series temporales, la validación siempre debe ser cronológica, nunca aleatoria.

**Entregables finales:**

* Archivo `.ipynb` con el cuaderno Jupyter completo y el código comentado.
* Archivo `.html` exportado desde Jupyter (Archivo > Descargar como > HTML).
* Archivo `.pdf` exportado desde Jupyter (Archivo > Descargar como > PDF).
* Presentación con 5 diapositivas para la exposición final.

---

**Fuentes de datos recomendadas:**

| Organismo | Datos | URL |
|---|---|---|
| INE (Instituto Nacional de Estadística) | PIB, IPC, EPA (desempleo), producción industrial | https://www.ine.es |
| Banco de España | Tipos de interés, crédito, deuda, euríbor | https://www.bde.es |
| Eurostat | Datos armonizados UE, PMI, balanza comercial | https://ec.europa.eu/eurostat |
| S&P Global / Markit | PMI Manufacturero y Servicios España | https://www.spglobal.com |
| Tesoro Público | Tipos de la deuda española (bonos 2Y, 10Y) | https://www.tesoro.es |
| Yahoo Finance | IBEX 35 (^IBEX), precio del oro, divisas | https://finance.yahoo.com |
| Ministerio de Trabajo | Afiliaciones a la Seguridad Social | https://www.mites.gob.es |

<h2 style="color:darkcyan;">0. Glosario de variables: qué estamos midiendo y por qué</h2>

Antes de escribir una sola línea de código es imprescindible entender qué mide cada variable, en qué unidades está y qué organismo la publica. Un modelo bien fundamentado económicamente es más útil —y más interpretable— que uno que optimiza métricas a ciegas.

---

### Variables de actividad económica

| Variable | Nombre completo | Descripción | Interpretación |
|---|---|---|---|
| **PIB** | Producto Interior Bruto | Valor total de bienes y servicios producidos en España en un período. Publicado trimestralmente por el INE. | Crece en expansión, decrece en recesión. Dos trimestres consecutivos de caída definen una recesión técnica. |
| **PIB_var** | Variación intertrimestral del PIB | Variación porcentual del PIB respecto al trimestre anterior, en términos reales (deflactado). | Positiva en expansión; negativa en recesión. Es la variable objetivo principal del modelo. |
| **IPI** | Índice de Producción Industrial | Mide la evolución mensual de la producción en las ramas industriales. Publicado por el INE. | Indicador adelantado de la actividad manufacturera. Cae antes de que el PIB lo haga. |

---

### Indices de actividad empresarial

| Variable | Nombre completo | Descripción | Interpretación |
|---|---|---|---|
| **PMI Manufacturero** | Purchasing Managers' Index — Manufactura España | Encuesta mensual a directores de compras del sector industrial. Elaborada por S&P Global / Markit. | Por encima de 50: el sector se expande. Por debajo de 50: se contrae. Anticipa en 1-2 meses la dirección del PIB. |
| **PMI Servicios** | Purchasing Managers' Index — Servicios España | La misma encuesta para el sector servicios (turismo, comercio, finanzas). Los servicios representan el 75% del PIB español. | Por encima de 50: expansión. Por debajo de 50: contracción. Especialmente relevante para la economía española por el peso del turismo. |

---

### Mercado laboral

| Variable | Nombre completo | Descripción | Interpretación |
|---|---|---|---|
| **Tasa de paro** | Tasa de desempleo (EPA) | Porcentaje de la población activa que busca trabajo y no lo encuentra. Publicada trimestralmente por el INE mediante la Encuesta de Población Activa (EPA). | Sube durante recesiones con retraso de 1-2 trimestres respecto al PIB. Es un indicador rezagado. |
| **Afiliaciones SS** | Afiliaciones a la Seguridad Social | Número de trabajadores dados de alta. Publicado mensualmente por el Ministerio de Trabajo. | Indicador adelantado más fiable que la EPA. Cae rápidamente cuando la actividad se deteriora. |

---

### Inflacion y precios

| Variable | Nombre completo | Descripción | Interpretación |
|---|---|---|---|
| **IPC** | Indice de Precios al Consumidor | Mide la variación del precio de una cesta representativa de bienes y servicios en España. Publicado por el INE. | Por encima del 2% anual: presión inflacionaria que puede llevar al BCE a subir tipos. Por debajo de 0%: riesgo de deflación. |
| **IPCA** | Indice de Precios al Consumo Armonizado | Version del IPC calculada con metodología común para todos los países de la UE. Publicado por Eurostat. | Permite comparar la inflación española con la del resto de la zona euro. |

---

### Tipos de interes y mercado de deuda

| Variable | Nombre completo | Descripción | Interpretación |
|---|---|---|---|
| **Tipo BCE** | Tipo de interes de referencia del BCE | Tipo al que el Banco Central Europeo presta dinero a los bancos comerciales. Fijado por el BCE. | Sube para combatir la inflación; baja para estimular la economía. Afecta directamente a hipotecas y crédito en España. |
| **Euribor 12M** | Euro Interbank Offered Rate a 12 meses | Tipo al que los bancos europeos se prestan dinero entre sí. Publicado por el Banco de España y Eurostat. | Referencia principal de las hipotecas variables en España. Anticipa la presión financiera sobre los hogares. |
| **Prima de riesgo** | Diferencial bono español 10Y - bono aleman 10Y | Diferencia entre la rentabilidad del bono español a 10 años y el aleman (Bund). | Cuando sube, indica que el mercado percibe mayor riesgo en la deuda española. Por encima de 300 pb: zona de tensión. |
| **Curva de tipos** | Diferencial bono 10Y - bono 2Y (España) | Diferencia entre el tipo a largo plazo y el tipo a corto plazo de la deuda soberana española. | Cuando es negativa (curva invertida) ha sido señal histórica de recesión con 12-18 meses de antelación. |

---

### Mercados financieros

| Variable | Nombre completo | Descripción | Interpretación |
|---|---|---|---|
| **IBEX 35** | Indice de la Bolsa española | Indice bursátil que agrupa las 35 empresas más liquidas de la Bolsa de Madrid. Ticker: ^IBEX en Yahoo Finance. | Indicador adelantado: cae antes de que el PIB lo haga y sube antes de la recuperación. |
| **Precio del oro** | Precio del oro en dolares por onza troy | Activo refugio cuyo precio sube cuando la incertidumbre económica aumenta. | Subidas sostenidas del oro pueden indicar mayor aversión al riesgo global. |

---

### Variable objetivo

| Fase del ciclo | Condicion de asignacion | Descripcion economica |
|---|---|---|
| **Expansion** | PIB_var > 0.6% trimestral Y PMI > 52 | Economía creciendo con fuerza; desempleo en mínimos, consumo e inversión robustos. |
| **Desaceleracion** | PIB_var > 0% pero decreciente, PMI entre 48-52 | Crecimiento positivo pero perdiendo impulso. Señal de advertencia temprana. |
| **Recesion** | PIB_var < 0% durante 2 o mas trimestres consecutivos | Contracción de la actividad. Desempleo en alza, consumo e inversión cayendo. |
| **Recuperacion** | PIB_var acelerándose desde negativo, PMI > 50 | La economía toca fondo y vuelve a crecer. El desempleo sigue alto pero deja de subir. |

---

### Variables derivadas (las crearás en la Parte 2)

| Variable | Como se calcula | Para que sirve |
|---|---|---|
| **pmi_diff** | PMI actual menos su media movil de 3 meses | Captura si el PMI está por encima o por debajo de su tendencia reciente. |
| **pmi_trend** | Pendiente lineal de los últimos 3 meses del PMI | Indica si el PMI lleva meses mejorando o empeorando. |
| **spread_tipos** | Euribor 12M menos tipo BCE | Mide la tensión en el mercado interbancario. |
| **lags t-1, t-3, t-6** | Valor de una variable en meses anteriores | Introducen memoria temporal: lo ocurrido hace 3 o 6 meses influye en el presente. |

<h2 style="color:darkcyan;">Parte 1: Preparación y Exploración de Datos (20%)</h2>

<h3 style="color:darkcyan;">1.1 Importación de librerías</h3>

Importa todas las librerías necesarias para el proyecto. Organízalas en bloques temáticos y añade un comentario breve junto a cada una que explique para qué la utilizarás.

In [10]:
import os
# import gdown
import shutil
from pathlib import Path
import re
import unicodedata # Added for normalizing column names

# Manipulación y análisis de datos
import pandas as pd
import numpy as np
import math

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.impute import SimpleImputer
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# Modelo de boosting
import xgboost as xgb

# Series temporales
from statsmodels.tsa.stattools import adfuller

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

<h3 style="color:darkcyan;">1.2 Carga y primera inspección del dataset</h3>

Carga los datos que hayas recopilado de las fuentes indicadas en la portada. Antes de continuar, responde en una celda de texto a las siguientes preguntas:

- ¿Qué periodo temporal cubre el dataset?
- ¿Cuál es la frecuencia de los datos: mensual, trimestral, anual?
- ¿Hay variables medidas con frecuencias distintas? ¿Cómo lo resolverás?

Muestra las primeras filas del dataset y un resumen estadístico básico.

---

**Respuestas a las preguntas:**

- **Periodo temporal cubierto:** El dataset combinado cubre un periodo que va desde el `1961-01-01` hasta el `2026-04-01`, según las fechas mínimas y máximas de las distintas series. Sin embargo, no todas las variables tienen datos para todo este rango, y la mayoría se concentran a partir de los años 2000.

- **Frecuencia de los datos:** Las variables tienen distintas frecuencias:
    - **Mensual:** PMI Manufacturero, PMI Servicios, Afiliaciones SS, Bono 10Y, Bono 2Y, IBEX 35, Euribor 12M, Tipo BCE, IPC, IPCA, IPI, Precio Oro, Prima de Riesgo, Curva de Tipos.
    - **Trimestral:** PIB (y PIB_var), Tasa de Paro (EPA).

- **Variables con frecuencias distintas:** Sí, existen variables con frecuencias mensuales y trimestrales. Para resolver esto, en la fase de limpieza y preparación de datos, se consolidará el dataset a una única frecuencia (probablemente mensual o trimestral, dependiendo del objetivo principal del modelo y de la variable objetivo), o se utilizarán técnicas de remuestreo e interpolación para alinear todas las series a una frecuencia común. Por ejemplo, las variables trimestrales pueden ser interpoladas para obtener valores mensuales, o las variables mensuales pueden ser promediadas/resampleadas para obtener valores trimestrales.

In [ ]:
# Descarga de datasets
# Navigate to data directory (supports both local and Docker execution)
local_path = Path("../../../data").resolve()
docker_path = Path("/data")

if docker_path.exists():
    DATA_DIR = docker_path
elif local_path.exists():
    DATA_DIR = local_path
else:
    DATA_DIR = local_path  # Default to local path
    os.makedirs(DATA_DIR, exist_ok=True)

print(f"Data directory: {DATA_DIR}")

# gdown.download_folder(
#     id="1p1GQad2f68ghLysO9Og0-YlY8onainTF",
#     output='data',
#     quiet=False,
#     use_cookies=False
# )

<h4 style="color:darkcyan;">PMI Manufactureo</h4>

In [12]:
pmi_m = pd.read_csv(DATA_DIR / "pmi_manufacturero_espana_mensual.csv", sep="\t")
pmi_m["fecha"] = pd.to_datetime(pmi_m["Date"], format="%Y.%m.%d", errors="coerce")
pmi_m["PMI_Manufacturero"] = pd.to_numeric(pmi_m["ActualValue"], errors="coerce")
pmi_m_clean = pmi_m[["fecha", "PMI_Manufacturero"]]
pmi_m_clean.head()

FileNotFoundError: [Errno 2] No such file or directory: '/data/pmi_manufacturero_espana_mensual.csv'

<h4 style="color:darkcyan;">PMI Servicios</h4>

In [ ]:
pmi_s = pd.read_csv(DATA_DIR / "pmi_servicios_espana_mensual.csv", sep="\t")
pmi_s["fecha"] = pd.to_datetime(pmi_s["Date"], format="%Y.%m.%d", errors="coerce")
pmi_s["PMI_Servicios"] = pd.to_numeric(pmi_s["ActualValue"], errors="coerce")
pmi_s_clean = pmi_s[["fecha", "PMI_Servicios"]]
pmi_s_clean.head()

<h4 style="color:darkcyan;">Afiliaciones SS</h4>

In [ ]:
af = pd.read_csv(DATA_DIR / "afiliaciones_ss_medias.csv", sep=";", encoding="latin1", skiprows=1)

col_edad = next(c for c in af.columns if "Tramos" in c)
col_regimen = next(c for c in af.columns if "Reg" in c)
col_sexo = next(c for c in af.columns if "Sexo" in c)

af = af[
    af[col_edad].astype(str).str.strip().eq("Total Edad")
    & af[col_regimen].astype(str).str.contains("Todos los Regimen", case=False, na=False)
    & af[col_sexo].astype(str).str.strip().eq("Total Sexo")
].copy()

af["fecha"] = pd.to_datetime(af["Periodo"].astype(str) + "01", format="%Y%m%d", errors="coerce")
af["afiliaciones_ss"] = pd.to_numeric(
    af["Afiliados medios"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)

af_clean = af[["fecha", "afiliaciones_ss"]].sort_values("fecha").reset_index(drop=True)
af_clean.head()

<h4 style="color:darkcyan;">Bono 10Y</h4>

In [ ]:
b10 = pd.read_csv(DATA_DIR / "bono_es_10y_rendimiento_mensual.csv")
b10["fecha"] = pd.to_datetime(b10["Fecha"], format="%d.%m.%Y", errors="coerce")
b10["bono_10y"] = pd.to_numeric(
    b10["Último"].astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
    errors="coerce"
)
b10_clean = b10[["fecha", "bono_10y"]]
b10_clean.head()

<h4 style="color:darkcyan;">Bono 2Y</h4>

In [ ]:
b2 = pd.read_csv(DATA_DIR / "bono_es_2y_rendimiento_mensual.csv")
b2["fecha"] = pd.to_datetime(b2["Fecha"], format="%d.%m.%Y", errors="coerce")
b2["bono_2y"] = b2["Último"].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float)
b2_clean = b2[["fecha", "bono_2y"]]
b2_clean.head()

<h4 style="color:darkcyan;">IBEX 35</h4>

In [ ]:
ibex = pd.read_csv(DATA_DIR / "ibex_35_historico_mensual.csv")

ibex["fecha"] = pd.to_datetime(ibex["Fecha"], format="%d.%m.%Y", errors="coerce")
ibex["ibex_35"] = pd.to_numeric(
    ibex["Último"].astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
    errors="coerce"
)

ibex_clean = ibex[["fecha", "ibex_35"]]
ibex_clean.head()

<h4 style="color:darkcyan;">Euribor 12M</h4>

In [ ]:
eur = pd.read_csv(DATA_DIR / "euribor_12m_bde.csv", encoding="latin1")

col_fecha = eur.columns[0]
mes = {"ENE": 1, "FEB": 2, "MAR": 3, "ABR": 4, "MAY": 5, "JUN": 6, "JUL": 7, "AGO": 8, "SEP": 9, "OCT": 10, "NOV": 11, "DIC": 12}

# Filtrar filas con patrón MES AÑO (ej: "ENE 2020")
eur = eur[eur[col_fecha].astype(str).str.contains(r"^[A-ZÑ]{3}\s+\d{4}$", na=False)].copy()
eur["fecha"] = eur[col_fecha].apply(lambda x: pd.Timestamp(int(str(x).split()[-1]), mes[str(x).split()[0]], 1))

# Euribor 12M
eur["euribor_12m"] = pd.to_numeric(
    eur["D_1NBAF472"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)

eur_clean = eur[["fecha", "euribor_12m"]]
eur_clean.head()

<h4 style="color:darkcyan;">Tipos BCE</h4>

In [ ]:
tip = pd.read_csv(DATA_DIR / "tipos_uem_bce_euribor_mensual.csv", encoding="latin1")
col = tip.columns[0]
tip_clean = tip[tip[col].astype(str).str.match(r"^[A-ZÑ]{3}\s+\d{4}$", na=False)].copy()

mes = {"ENE": 1, "FEB": 2, "MAR": 3, "ABR": 4, "MAY": 5, "JUN": 6, "JUL": 7, "AGO": 8, "SEP": 9, "OCT": 10, "NOV": 11, "DIC": 12}
tip_clean["fecha"] = tip_clean[col].apply(lambda x: pd.Timestamp(int(x.split()[-1]), mes[x.split()[0]], 1))

for col_name, new_name in [("D_1TFK09A0_BCE", "tipo_bce"), ("D_1NBAF472", "euribor_12m")]:
    if col_name in tip_clean.columns:
        tip_clean[new_name] = pd.to_numeric(
            tip_clean[col_name].astype(str).str.replace(",", ".", regex=False),
            errors="coerce"
        )
    else:
        tip_clean[new_name] = np.nan

tip_clean["spread_tipos"] = tip_clean["euribor_12m"] - tip_clean["tipo_bce"]
tip_clean[["fecha", "tipo_bce", "euribor_12m", "spread_tipos"]].head()

<h4 style="color:darkcyan;">IPC</h4>

In [ ]:
raw = pd.read_csv(DATA_DIR / "ipc_ine.csv", header=None, dtype=str)
h = [i for i, r in raw.iterrows() if "M01" in r.values and "M12" in r.values][0]
m_labels = raw.loc[h].tolist()
recs = []
for i in range(h + 1, len(raw)):
    y = str(raw.iloc[i, 1]).strip()
    if re.match(r"^\d{4}$", y):
        for j, m in enumerate(m_labels):
            if re.match(r"^M(\d{2})", str(m)):
                recs.append({"fecha": pd.Timestamp(int(y), int(str(m)[1:3]), 1), "ipc_mensual_var": raw.iloc[i, j]})
ipc_clean = pd.DataFrame(recs)
ipc_clean["ipc_mensual_var"] = pd.to_numeric(
    ipc_clean["ipc_mensual_var"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)
ipc_clean.head()

<h4 style="color:darkcyan;">IPCA</h4>

In [ ]:
raw = pd.read_csv(DATA_DIR / "ipca_eurostat.csv", header=None, dtype=str)
h = [i for i, r in raw.iterrows() if sum(bool(re.match(r"^\d{4}M\d{2}$", str(v).strip())) for v in r.fillna("")) >= 24][0]
v = [i for i, r in raw.iterrows() if any("indice" in str(cell).lower() or "índice" in str(cell).lower() for cell in r.fillna("") if "general" in str(cell).lower())][0]
m_row, v_row = raw.loc[h], raw.loc[v]
recs = []
for j in range(len(m_row)):
    token = str(m_row.iloc[j]).strip()
    if re.match(r"^\d{4}M\d{2}$", token):
        recs.append({"fecha": pd.Timestamp(int(token[:4]), int(token[5:7]), 1), "ipca_var_anual": v_row.iloc[j]})
ipca_clean = pd.DataFrame(recs)
ipca_clean["ipca_var_anual"] = pd.to_numeric(ipca_clean["ipca_var_anual"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
ipca_clean = ipca_clean.dropna(subset=["ipca_var_anual"])
ipca_clean.head()

<h4 style="color:darkcyan;">PIB trimestral (PIB y PIB_var)</h4>

In [ ]:
pib = pd.read_csv(DATA_DIR / "pib_trimestral_ine.csv", header=None, dtype=str)
h = [i for i, r in pib.iterrows() if sum(str(v).strip() in {"T1", "T2", "T3", "T4"} for v in r.fillna("")) >= 4][0]

recs = []
for i in range(h + 1, len(pib)):
    y = str(pib.iloc[i, 1]).strip()
    if re.match(r"^\d{4}$", y):
        for j in range(4):
            recs.append({"fecha": pd.Timestamp(int(y), j * 3 + 1, 1), "PIB_var": pib.iloc[i, j + 2]})

pib_clean = pd.DataFrame(recs)
pib_clean["PIB_var"] = pd.to_numeric(pib_clean["PIB_var"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
pib_clean = pib_clean.sort_values("fecha").reset_index(drop=True)

# Serie de nivel de PIB (índice encadenado base 100) a partir de la variación trimestral
pib_clean["PIB"] = 100 * (1 + pib_clean["PIB_var"] / 100).cumprod()

pib_clean[["fecha", "PIB", "PIB_var"]].head()

<h4 style="color:darkcyan;">IPI (Índice de Producción Industrial)</h4>

In [ ]:
ipi = pd.read_csv(DATA_DIR / "indice_produccion_industrial_ine.csv", sep=";", dtype=str)
ipi = ipi[
    (ipi["Comunidades y Ciudades Autónomas"].astype(str).str.strip() == "Total Nacional")
    & (ipi["Destino económico de los bienes"].astype(str).str.strip() == "Total industria")
    & (ipi["Índice y tasas"].astype(str).str.strip() == "Índice")
].copy()

ipi["fecha"] = pd.to_datetime(ipi["Periodo"].astype(str), format="%YM%m", errors="coerce")
ipi["IPI"] = pd.to_numeric(ipi["Total"].astype(str).str.replace(",", ".", regex=False), errors="coerce")

ipi_clean = ipi[["fecha", "IPI"]]
ipi_clean.head()

<h4 style="color:darkcyan;">Tasa de paro (EPA)</h4>

In [ ]:
paro = pd.read_csv(DATA_DIR / "tasa_paro_epa_total_nacional.csv", header=None, dtype=str)

h = [
    i for i, r in paro.iterrows()
    if sum(bool(re.match(r"^\d{4}T\d$", str(v).strip())) for v in r.fillna("")) >= 4
][0]
row_labels = paro.iloc[:, 0].astype(str).str.strip()
v = row_labels[row_labels.eq("Total Nacional")].index[0]

q_row = paro.loc[h]
val_row = paro.loc[v]
group_row = paro.loc[h - 1]  # fila con bloques: Total, Menores de 25 años, ...

group_starts = [j for j, val in enumerate(group_row) if str(val).strip() in {"Total", "Menores de 25 años", "25 y más años", "De 16 a 19 años", "De 20 a 24 años", "De 25 a 54 años", "De 55 y más años"}]
start_total = group_starts[0]
end_total = group_starts[1] if len(group_starts) > 1 else len(q_row)

recs = []
for j in range(start_total, end_total):
    token = str(q_row.iloc[j]).strip()
    if re.match(r"^\d{4}T\d$", token):
        year = int(token[:4])
        quarter = int(token[5])
        recs.append({"fecha": pd.Timestamp(year, (quarter - 1) * 3 + 1, 1), "tasa_paro": val_row.iloc[j]})

paro_clean = pd.DataFrame(recs)
paro_clean["tasa_paro"] = pd.to_numeric(paro_clean["tasa_paro"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
paro_clean = paro_clean.sort_values("fecha").reset_index(drop=True)
paro_clean.head()

<h4 style="color:darkcyan;">Precio del oro</h4>

In [ ]:
oro = pd.read_csv(DATA_DIR / "precio_oro_historico_mensual.csv")
oro["fecha"] = pd.to_datetime(oro["Fecha"], format="%d.%m.%Y", errors="coerce")
oro["precio_oro"] = pd.to_numeric(
    oro["Último"].astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
    errors="coerce"
)

oro_clean = oro[["fecha", "precio_oro"]]
oro_clean.head()

<h4 style="color:darkcyan;">Prima de riesgo</h4>

In [ ]:
deuda = pd.read_csv(DATA_DIR / "deuda_publica_ibex_diferenciales_mensual.csv", encoding="latin1")
col_fecha_deuda = deuda.columns[0]
mes = {"ENE": 1, "FEB": 2, "MAR": 3, "ABR": 4, "MAY": 5, "JUN": 6, "JUL": 7, "AGO": 8, "SEP": 9, "OCT": 10, "NOV": 11, "DIC": 12}

# El fichero viene en formato diario laborable tipo "03 ENE 2000"
deuda = deuda[deuda[col_fecha_deuda].astype(str).str.contains(r"^\d{2}\s+[A-ZÑ]{3}\s+\d{4}$", na=False)].copy()

def parse_bde_day_month_year(x: str) -> pd.Timestamp:
    parts = str(x).split()
    day = int(parts[0])
    month = mes[parts[1]]
    year = int(parts[2])
    return pd.Timestamp(year, month, day)

deuda["fecha"] = deuda[col_fecha_deuda].apply(parse_bde_day_month_year)
deuda["prima_riesgo"] = pd.to_numeric(deuda["DPUDNBBN304D"].replace("_", np.nan), errors="coerce")

# Resample mensual para alinear con el resto de series
prima_riesgo_clean = (
    deuda[["fecha", "prima_riesgo"]]
    .dropna(subset=["prima_riesgo"])
    .set_index("fecha")
    .resample("MS")
    .mean()
    .reset_index()
)

prima_riesgo_clean.head()

<h4 style="color:darkcyan;">Curva de tipos (10Y - 2Y)</h4>

In [ ]:
curva_tipos_clean = (
    b10_clean.merge(b2_clean, on="fecha", how="inner")
    .assign(curva_tipos=lambda d: d["bono_10y"] - d["bono_2y"])
    [["fecha", "curva_tipos"]]
)

curva_tipos_clean.head()

In [ ]:
df = pmi_m_clean

df = df.merge(pmi_s_clean, on='fecha', how='outer')
df = df.merge(af_clean, on='fecha', how='outer')
df = df.merge(b10_clean, on='fecha', how='outer')
df = df.merge(b2_clean, on='fecha', how='outer')
df = df.merge(ibex_clean, on='fecha', how='outer')
df = df.merge(eur_clean, on='fecha', how='outer')
df = df.merge(tip_clean[['fecha', 'tipo_bce', 'spread_tipos']], on='fecha', how='outer')
df = df.merge(ipc_clean, on='fecha', how='outer')
df = df.merge(ipca_clean, on='fecha', how='outer')
df = df.merge(pib_clean, on='fecha', how='outer')
df = df.merge(ipi_clean, on='fecha', how='outer')
df = df.merge(paro_clean, on='fecha', how='outer')
df = df.merge(oro_clean, on='fecha', how='outer')
df = df.merge(prima_riesgo_clean, on='fecha', how='outer')
df = df.merge(curva_tipos_clean, on='fecha', how='outer')

# Ordenar por fecha
df = df.sort_values('fecha').reset_index(drop=True)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

<h3 style="color:darkcyan;">1.3 Limpieza de datos</h3>

Realiza las siguientes tareas y justifica cada decisión tomada:

**Valores faltantes:** Identifica cuántos valores NA tiene cada columna y qué porcentaje representan. Para cada variable, elige una estrategia de imputación (media, interpolación lineal, eliminación de filas) y razona por qué es la más adecuada para ese tipo de dato.

**Outliers:** Utiliza el método IQR para detectar valores atípicos. Para cada outlier identificado, decide si eliminarlo, recortarlo o mantenerlo, teniendo en cuenta si puede corresponder a un evento histórico real (por ejemplo, la crisis de 2008 o el impacto del COVID-19 en 2020).

**Calidad temporal:** Verifica que la columna de fecha está en formato `datetime`, que los datos están ordenados cronológicamente y que no existen saltos inesperados en la serie.

In [ ]:
# Análisis de valores faltantes
# Calcula el número total de valores nulos por columna
missing_counts = df.isnull().sum()

# Calcula el porcentaje de valores nulos por columna
missing_percentages = (df.isnull().sum() / len(df)) * 100

# Crea un DataFrame para mostrar los resultados de manera organizada
missing_info = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing Percentage': missing_percentages
})

# Filtra las columnas que tienen al menos un valor faltante y ordénalas
missing_info = missing_info[missing_info['Missing Count'] > 0].sort_values(by='Missing Percentage', ascending=False)

print("Valores faltantes por columna:")
display(missing_info)

Para manejar las distintas frecuencias de los datos (mensuales y trimestrales) y asegurar la consistencia temporal, primero normalizamos todas las fechas a principios de mes y luego reindexamos el DataFrame a una frecuencia mensual completa. Si hay múltiples entradas para el mismo mes, tomamos el último valor no nulo disponible para ese mes.

In [ ]:
# Normalización de fecha a principios de mes
df["fecha"] = pd.to_datetime(df["fecha"])
df["fecha"] = df["fecha"].dt.to_period("M").dt.to_timestamp(how="start")

# Identificación de columnas numéricas
num_cols = df.select_dtypes(include=np.number).columns.tolist()

# Resolución de duplicados por mes, se prioriza el último valor no nulo
def last_valid(s):
    s_valid = s.dropna()
    return s_valid.iloc[-1] if len(s_valid) > 0 else np.nan

df = (
    df.sort_values("fecha")
      .groupby("fecha", as_index=False)[num_cols]
      .agg(last_valid)
      .sort_values("fecha")
)

# Reindex a calendario mensual completo para rellenar los meses intermedios
full_months = pd.date_range(
    start=df["fecha"].min(),
    end=df["fecha"].max(),
    freq="MS"
)

df = (
    df.set_index("fecha")
      .reindex(full_months)
      .rename_axis("fecha")
      .reset_index()
)

display(df.head())

Se va ha limitar el dataset a 2002 porque es un intervalo de tiempo en el que la mayoría de las variables están disponibles, para así mejorar la fiabilidad del modelo.

Se ha elegido 2002 porque responde a que muchas de las variables incluidas (como el Euríbor, los tipos del BCE, los índices PMI o determinados indicadores financieros) no estaban disponibles o no se recopilaban de forma homogénea en España en décadas anteriores.

In [ ]:
# Recortar filas
df['fecha'] = pd.to_datetime(df['fecha'])
df = df[df['fecha'] >= '2002-01-01'].reset_index(drop=True)

df.head()

Una vez que el DataFrame tiene una frecuencia mensual uniforme, rellenamos los valores faltantes. Vamos a utilizar `ffill()` (forward-fill) para propagar el último valor válido conocido hacia adelante ya que evita el uso de información futura (`data leakage`) y mantiene la coherencia temporal de las series.

In [ ]:
# Imputación de valores faltantes
df[num_cols] = df[num_cols].ffill()

df.head()

In [ ]:
# Descargar el dataset final
import os

# Asegurarse de que el directorio de destino exista
os.makedirs('data/processed', exist_ok=True)

# Guardar el DataFrame 'df' a un archivo CSV
df.to_csv('data/processed/df_final.csv', index=False)

print("DataFrame 'df' guardado exitosamente en 'data/processed/df_final.csv'")

In [ ]:
# Detección de outliers con el método IQR
# Recuerda: un valor es outlier si está por debajo de Q1 - 1.5*IQR o por encima de Q3 + 1.5*IQR

for column in df.select_dtypes(include=np.number).columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]

    if not outliers.empty:
        print(f"\nOutliers detectados en la columna '{column}':")
        print(outliers[['fecha', column]])
    else:
        print(f"\nNo se detectaron outliers en la columna '{column}'.")

Por el momento vamos a dejar los outliers, ya que en el contexto de algunas variables esos valores corresponden a eventos económicos importantes reales.

In [ ]:
# Validacion visual de outliers (IQR) por variable numerica
num_cols = df.select_dtypes(include=np.number).columns.tolist()
n_cols = 3
n_rows = int(np.ceil(len(num_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows), sharex=False)
axes = np.array(axes).reshape(-1)

for i, col in enumerate(num_cols):
    ax = axes[i]
    serie = df[["fecha", col]].dropna().copy()

    if serie.empty:
        ax.set_title(col)
        ax.text(0.5, 0.5, "Sin datos", ha="center", va="center", transform=ax.transAxes)
        continue

    q1 = serie[col].quantile(0.25)
    q3 = serie[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr

    mask_out = (serie[col] < low) | (serie[col] > high)

    ax.plot(serie["fecha"], serie[col], linewidth=1.2, alpha=0.8)
    ax.scatter(serie.loc[mask_out, "fecha"], serie.loc[mask_out, col], s=14, color="red", alpha=0.85)
    ax.set_title(col)
    ax.grid(True, alpha=0.25)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Series temporales con outliers detectados (IQR)", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

# Resumen de cantidad de outliers por variable
outlier_counts = []
for col in num_cols:
    s = df[col].dropna()
    if s.empty:
        outlier_counts.append({"variable": col, "n_outliers": 0, "pct_outliers": 0.0})
        continue
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    n_out = int(((s < low) | (s > high)).sum())
    outlier_counts.append({
        "variable": col,
        "n_outliers": n_out,
        "pct_outliers": round(100 * n_out / len(s), 2)
    })

outlier_summary = pd.DataFrame(outlier_counts).sort_values(["n_outliers", "pct_outliers"], ascending=False)
display(outlier_summary[outlier_summary["n_outliers"] > 0])

### Análisis sobre las variables con outliers

| Variable | ¿Cuándo? | Valores | Explicación |
| :--- | :--- | :--- | :--- |
| **PMI Manufacturero** | Mayo-Junio 2020 | Cayó a 30.8 – 38.3 | La industria paró en seco por el confinamiento del COVID. |
| **PMI Servicios** | Abril-Junio 2020 | Se hundió a 7.1 | Hoteles, bares, viajes: todo cerrado por la pandemia. |
| **PMI Servicios** | 2011-2012 y 2020-2021 | Varios meses entre 36 y 42 | Crisis de deuda europea (2012) y rebrote COVID (2021). |
| **IBEX 35** | 2007 (varios meses) | Subió a 15.000 – 15.800 | Burbuja pre-crisis financiera. Máximos de la época. |
| **IBEX 35** | 2025-2026 (desde agosto 2025) | Subió hasta 18.467 y se mantuvo | Los bancos ganaron más (tipos altos) y España dio confianza. |
| **IPC (inflación)** | Enero de varios años (2009-2019) | Cayó a -1.2 / -1.9 | Inflación negativa (deflación) típica de enero por rebajas. |
| **IPC (inflación)** | Marzo 2022 | Subió a +3.0% | Guerra de Ucrania → disparó precios de energía y alimentos. |
| **IPI (producción industrial)** | 2006-2008 | Valores altos (~140) | La industria iba a toda máquina antes de la crisis. |
| **IPI (producción industrial)** | 2013-2014 | Cayó a ~74 | Crisis profunda: fábricas cerradas o produciendo muy poco. |
| **IPI (producción industrial)** | Abril 2020 | Cayó a 66 (mínimo histórico) | COVID: fábricas paradas. |
| **Precio del oro** | 2025-2026 (desde marzo 2025) | Explotó de 3.150 → 5.267 $ | Los inversores buscaron refugio por miedo al dólar y la deuda. |
| **Prima de riesgo** | 2024-2025 (varios meses) | Cayó a mínimos (2.1%) | España se percibía como muy segura frente a Alemania. |
| **PIB (crecimiento)** | 2008-2009 | Cayó a -2.7% | Crisis financiera global. |
| **PIB (crecimiento)** | 2012 | Cayó a -0.9% | Segunda recesión (crisis de deuda europea). |
| **PIB (crecimiento)** | Abril-Junio 2020 | Se desplomó a -17.8% | COVID: economía paralizada. |
| **PIB (crecimiento)** | Julio-Septiembre 2020 | Rebote a +15.9% | Reapertura tras el confinamiento (efecto rebote). |

**Cada outlier marca una crisis o un cambio de época:**
- **2008-2012** → Crisis financiera + deuda europea.
- **2020** → El COVID paró el mundo (PMI, PIB, IPI por los suelos).
- **2022** → Guerra de Ucrania disparó la inflación.
- **2025-2026** → Subida de tipos + confianza en España + oro por las nubes = IBEX y oro en máximos históricos.

<h3 style="color:darkcyan;">1.4 Análisis Exploratorio de Datos (EDA)</h3>

Representa gráficamente las series temporales de las variables más relevantes. En cada gráfico, sombrea los períodos históricos de recesión española reconocidos para facilitar la interpretación visual:

- Crisis financiera global: 2008T4 – 2013T2
- Impacto COVID-19: 2020T1 – 2020T2

Genera al menos los siguientes gráficos e interpreta cada uno:

1. Evolución del PIB trimestral español (variación intertrimestral)
2. PMI Manufacturero y de Servicios en España
3. Tasa de paro según la EPA
4. IPC interanual
5. Euribor 12 meses y tipo de referencia del BCE

A continuación, calcula y visualiza la **matriz de correlación** entre todas las variables numéricas. Identifica los pares de variables más correlacionados e interpreta si esa correlación tiene sentido económico.

In [ ]:
# Períodos de recesión española para sombrear en los gráficos
recesiones_españa = [
    ('2008-10-01', '2013-06-30'),   # Gran Recesión y crisis de deuda soberana
    ('2020-01-01', '2020-06-30'),   # COVID-19
]

def sombrear_recesiones(ax):
    """Añade franjas grises en los períodos de recesión española."""
    for inicio, fin in recesiones_españa:
        ax.axvspan(pd.to_datetime(inicio), pd.to_datetime(fin),
                   alpha=0.15, color='red')

# Utiliza esta función en tus gráficos llamando sombrear_recesiones(ax) después de crear cada subplot

In [ ]:
# Gráficos de series temporales

# 1. Evolución del PIB trimestral español (variación intertrimestral)
plt.figure(figsize=(15, 5))
sns.lineplot(data=df, x='fecha', y='PIB_var')
plt.title('1. Evolución del PIB Trimestral (Variación Intertrimestral)')
plt.ylabel('PIB Var. (%)')
sombrear_recesiones(plt.gca())
plt.xlabel('Fecha')
plt.tight_layout()
plt.show()

# 2. PMI Manufacturero y de Servicios en España
plt.figure(figsize=(15, 5))
sns.lineplot(data=df, x='fecha', y='PMI_Manufacturero', label='PMI Manufacturero')
sns.lineplot(data=df, x='fecha', y='PMI_Servicios', label='PMI Servicios')
plt.title('2. PMI Manufacturero y de Servicios en España')
plt.ylabel('PMI (puntos)')
plt.axhline(50, color='gray', linestyle='--', linewidth=0.8, label='Umbral 50 (Expansión/Contracción)')
plt.legend()
sombrear_recesiones(plt.gca())
plt.xlabel('Fecha')
plt.tight_layout()
plt.show()

# 3. Tasa de paro según la EPA
plt.figure(figsize=(15, 5))
sns.lineplot(data=df, x='fecha', y='tasa_paro')
plt.title('3. Tasa de Paro (EPA)')
plt.ylabel('Tasa de Paro (%)')
sombrear_recesiones(plt.gca())
plt.xlabel('Fecha')
plt.tight_layout()
plt.show()

# 4. IPC interanual (usando ipca_var_anual ya que es la armonizada e interanual)
plt.figure(figsize=(15, 5))
sns.lineplot(data=df, x='fecha', y='ipca_var_anual')
plt.title('4. IPCA (Índice de Precios al Consumo Armonizado) Anual')
plt.ylabel('IPCA Var. Anual (%)')
sombrear_recesiones(plt.gca())
plt.xlabel('Fecha')
plt.tight_layout()
plt.show()

# 5. Euribor 12 meses y tipo de referencia del BCE
plt.figure(figsize=(15, 5))
sns.lineplot(data=df, x='fecha', y='euribor_12m', label='Euribor 12M')
sns.lineplot(data=df, x='fecha', y='tipo_bce', label='Tipo Referencia BCE')
plt.title('5. Euribor 12 Meses y Tipo de Referencia del BCE')
plt.ylabel('Tipo de Interés (%)')
plt.legend()
sombrear_recesiones(plt.gca())
plt.xlabel('Fecha')
plt.tight_layout()
plt.show()

1. **Evolución del PIB Trimestral (Variación Intertrimestral):** Este gráfico muestra la variación porcentual del PIB respecto al trimestre anterior. Se observa una clara caída del PIB durante los periodos sombreados, correspondientes a la crisis financiera global (2008-2013) y el impacto del COVID-19 (2020). La recuperación después de la crisis de 2008 fue lenta y prolongada, mientras que tras el shock inicial del COVID-19, se produjo una recuperación en forma de 'V' en los trimestres posteriores, aunque no sin volatilidad.

2. **PMI Manufacturero y de Servicios en España:** Ambos índices PMI, tanto el manufacturero como el de servicios, tienden a caer significativamente por debajo del umbral de 50 (que marca la contracción) justo antes o durante los periodos de recesión. Esto confirma su naturaleza como indicadores adelantados o coincidentes de la actividad económica. En la crisis de 2008-2013, ambos mostraron periodos prolongados de contracción. Durante el COVID-19, se observó una caída muy abrupta seguida de una rápida recuperación.

3. **Tasa de Paro (EPA):** La tasa de paro, medida por la Encuesta de Población Activa (EPA), muestra un comportamiento rezagado con respecto a las recesiones. Durante la crisis financiera de 2008-2013, el desempleo escaló a niveles muy altos y permaneció elevado incluso después de que la economía comenzara a recuperarse. Con el COVID-19, se apreció un impacto inicial, pero las medidas como los ERTEs mitigaron un aumento aún mayor y más rápido. Su tendencia es inversamente proporcional al PIB: cuando el PIB cae, el paro sube, pero con un decalaje temporal.

4. **IPCA (Índice de Precios al Consumo Armonizado) Anual:** Este gráfico muestra la evolución de la inflación. Durante los periodos de recesión, la inflación tiende a moderarse o incluso a caer, reflejando una menor demanda y actividad económica. Se observan picos inflacionarios en otros periodos, como en 2008 antes de la crisis o más recientemente en 2021-2022, que pueden estar relacionados con factores de oferta o choques energéticos.

5. **Euribor 12 Meses y Tipo de Referencia del BCE:** Aquí se ve una estrecha relación entre el tipo de referencia del Banco Central Europeo y el Euribor a 12 meses, que es clave para las hipotecas. El BCE tiende a bajar los tipos de interés para estimular la economía durante las recesiones (como se ve después de 2008 y durante el COVID-19) y los sube para contener la inflación en periodos de crecimiento o de riesgo inflacionario. El Euribor sigue de cerca los movimientos del BCE, reflejando el coste del dinero para los bancos.

In [ ]:
# Matriz de correlación
# Pista: df.corr() calcula la matriz; usa sns.heatmap() con annot=True para visualizarla

correlation_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Matriz de Correlación de Variables Económicas')
plt.show()

<h2 style="color:darkcyan;">Parte 2: Ingeniería de Variables — Feature Engineering (25%)</h2>

En esta sección construimos nuevas variables predictoras a partir de las originales. La ingeniería de variables es uno de los pasos más importantes en Machine Learning: unas buenas variables diseñadas con criterio económico superan casi siempre a un modelo complejo con variables pobres.

<h3 style="color:darkcyan;">2.1 Variables derivadas</h3>

Crea las siguientes variables derivadas. En cada caso ya tienes la descripción en el glosario; el reto está en traducirla a código correctamente:

**Variables adelantadas basadas en el PMI:**
- `pmi_diff`: diferencia entre el PMI actual y su media móvil de 3 meses. Indica si el PMI está por encima o por debajo de su tendencia reciente.
- `pmi_trend`: pendiente de los últimos 3 meses del PMI. Puedes calcularla con `np.polyfit(range(3), valores, 1)[0]` dentro de un `rolling().apply()`.

**Variables de momentum económico:**
- `pib_var_anual`: variación interanual del PIB (comparando con el mismo trimestre del año anterior).
- `ipc_aceleracion`: segunda diferencia del IPC, es decir, si la inflación está acelerando o desacelerando.
- `paro_cambio`: variación trimestral de la tasa de paro.

**Variables rezagadas (lags):**

Crea versiones retrasadas de 1, 3 y 6 meses de al menos tres variables clave. Los lags introducen memoria temporal: lo que ocurrió hace varios meses influye en la situación actual.

**Variable de tension financiera:**
- `spread_tipos`: diferencia entre el Euribor 12M y el tipo de referencia del BCE. Mide la tension en el mercado interbancario.

In [ ]:
# Variables adelantadas basadas en el PMI

# pmi_diff: diferencia entre el PMI actual y su media móvil de 3 meses.
df['pmi_manufacturero_diff'] = df['PMI_Manufacturero'] - df['PMI_Manufacturero'].rolling(window=3).mean()
df['pmi_servicios_diff'] = df['PMI_Servicios'] - df['PMI_Servicios'].rolling(window=3).mean()

# pmi_trend: pendiente de los últimos 3 meses del PMI.
def calculate_trend(series):
    if len(series) == 3 and not series.isnull().any():
        return np.polyfit(range(3), series, 1)[0]
    return np.nan

df['pmi_manufacturero_trend'] = df['PMI_Manufacturero'].rolling(window=3).apply(calculate_trend, raw=False)
df['pmi_servicios_trend'] = df['PMI_Servicios'].rolling(window=3).apply(calculate_trend, raw=False)

print("Variables PMI calculadas:")
df[['fecha', 'PMI_Manufacturero', 'pmi_manufacturero_diff', 'pmi_manufacturero_trend', 'PMI_Servicios', 'pmi_servicios_diff', 'pmi_servicios_trend']].head()

En las dos primeras filas aparece nulo porque el diff y el trend se calcula con los tres meses anteriores.

In [ ]:
# Variables de momentum

# pib_var_anual: variación interanual del PIB (comparando con el mismo trimestre del año anterior).
df['pib_var_anual'] = df['PIB'].pct_change(periods=12) * 100

# ipc_aceleracion: segunda diferencia del IPC, es decir, si la inflación está acelerando o desacelerando.
df['ipc_aceleracion'] = df['ipc_mensual_var'].diff().diff()

# paro_cambio: variación trimestral de la tasa de paro.
df['paro_cambio'] = df['tasa_paro'].diff(periods=3)

df

In [ ]:
# Variables rezagadas
# Pista: df['variable'].shift(n) devuelve la variable con n períodos de retraso

variables_para_lag = ['PMI_Manufacturero', 'PMI_Servicios', 'PIB_var']   # indica aquí las columnas que quieres rezagar
lags = [1, 3, 6]

# Completa el bucle para crear todas las combinaciones
for col in variables_para_lag:
    for lag in lags:
        df[f'{col}_lag_{lag}'] = df[col].shift(lag)
df

In [ ]:
df.columns

<h3 style="color:darkcyan;">2.2 Variables adicionales propuestas por el equipo</h3>

Además de las variables descritas anteriormente, incorpora al menos dos variables propias que consideréis relevantes para predecir el ciclo económico español. Para cada una, completad la siguiente ficha antes de escribir el código:

---

**Variable propuesta 1:** `ibex_volatility`

- **Descripción**: La volatilidad mensual del IBEX 35, calculada como la desviación estándar móvil de los retornos logarítmicos diarios o mensuales del índice. Una mayor volatilidad indica una mayor incertidumbre y riesgo en el mercado bursátil.
- **Fuente de los datos**: Derivada de la columna `ibex_35` en el DataFrame `df`.
- **Justificación económica**: Los mercados bursátiles suelen ser indicadores adelantados de la economía. Un incremento en la volatilidad del índice IBEX 35 a menudo precede a periodos de desaceleración o recesión, ya que los inversores reaccionan de manera más brusca a la información económica en entornos de incertidumbre, aumentando las fluctuaciones de precios. Es una medida de la percepción de riesgo por parte del mercado.
- **Tipo de relación esperada con el ciclo**: Adelantada; relación negativa (alta volatilidad sugiere desaceleración/recesión).

---

**Variable propuesta 2:** `afiliaciones_ss_var_anual`

- **Descripción**: Variación porcentual interanual del número total de afiliados a la Seguridad Social. Este indicador ofrece una medida directa y mensual de la evolución del empleo, siendo más oportuno que la tasa de paro trimestral.
- **Fuente de los datos**: Derivada de la columna `afiliaciones_ss` en el DataFrame `df`.
- **Justificación económica**: Las afiliaciones a la Seguridad Social son un termómetro muy sensible de la creación o destrucción de empleo en España. Un crecimiento sostenido de las afiliaciones es característico de una fase de expansión, mientras que una desaceleración o caída en su tasa de variación interanual suele ser un signo temprano de un deterioro del mercado laboral y de la actividad económica general, incluso antes de que la tasa de paro trimestral lo refleje de forma contundente.
- **Tipo de relación esperada con el ciclo**: Coincidente/Ligeramente adelantada; relación positiva (mayor crecimiento de afiliaciones -> expansión).

---

Añade tantas fichas como variables incorporéis. La calidad de la justificación económica tiene tanto peso en la evaluación como el resultado técnico del modelo.

In [ ]:
# Código para cargar y calcular las variables adicionales propuestas

# Variable 1: Volatilidad del IBEX 35 (ibex_volatility)
# Calcularemos los retornos logarítmicos y luego la desviación estándar móvil.
df['ibex_returns'] = np.log(df['ibex_35'] / df['ibex_35'].shift(1))
df['ibex_volatility'] = df['ibex_returns'].rolling(window=12).std() * np.sqrt(12) # Volatilidad anualizada

# Variable 2: Variación Interanual de las Afiliaciones a la Seguridad Social (afiliaciones_ss_var_anual)
df['afiliaciones_ss_var_anual'] = df['afiliaciones_ss'].pct_change(periods=12) * 100

print("Variables adicionales calculadas:")
display(df[['fecha', 'ibex_35', 'ibex_volatility', 'afiliaciones_ss', 'afiliaciones_ss_var_anual']].head(15))

<h3 style="color:darkcyan;">2.3 Etiquetado de la variable objetivo</h3>

Crea la variable `cycle_phase` con las cuatro categorías del ciclo económico definidas en el glosario. Tienes a continuación el esqueleto de la función; debes completar la lógica de clasificación y aplicarla al dataset.

Una vez creada la variable objetivo, analiza la distribución de clases con un gráfico de barras. Si alguna clase tiene muy pocos ejemplos (menos del 10% del total), hay un **desbalanceo de clases** que deberás tratar más adelante.

In [ ]:
def clasificar_ciclo_español(row):
    """
    Asigna la fase del ciclo económico español en función de la
    variación intertrimestral del PIB y el PMI compuesto.

    Parámetros
    ----------
    row : fila del DataFrame con los campos 'PIB_var' y 'PMI_Manufacturero'.

    Devuelve
    --------
    str : 'Expansion', 'Desaceleracion', 'Recesion' o 'Recuperacion'.
    """
    pib = row.get('PIB_var', np.nan)
    pmi = row.get('PMI_Manufacturero', np.nan)

    if pd.isna(pib) or pd.isna(pmi):
        return np.nan

    if pib > 0.6 and pmi > 52:
        return 'Expansion'
    elif pib <= 0 and pmi > 50: # Recuperación: PIB aún no positivo pero PMI ya en expansión
        return 'Recuperacion'
    elif pib < 0: # Recesión: Contracción del PIB y no se cumplen las condiciones de recuperación
        return 'Recesion'
    elif pib > 0 and pib <= 0.6 and 48 <= pmi <= 52: # Desaceleración: Crecimiento lento y PMI en zona de estancamiento/desaceleración
        return 'Desaceleracion'
    else:
        return np.nan # Para casos que no encajen en ninguna categoría explícita


# Aplica la función y muestra la distribución de clases
df['cycle_phase'] = df.apply(clasificar_ciclo_español, axis=1)

In [ ]:
# Visualización de la distribución de clases
# Interpreta: ¿hay desbalanceo? ¿qué fase tiene menos ejemplos?

plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='cycle_phase', order=df['cycle_phase'].value_counts().index)
plt.title('Distribución de las Fases del Ciclo Económico')
plt.xlabel('Fase del Ciclo')
plt.ylabel('Número de Ejemplos')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("Número de ejemplos por fase:")
print(df['cycle_phase'].value_counts())

In [ ]:
# Guarda el dataset procesado antes de continuar
os.makedirs('data/processed', exist_ok=True)
df.to_csv('data/processed/dataset_espana_features.csv', index=False)

<h2 style="color:darkcyan;">Parte 3: Modelado y Evaluación (35%)</h2>

<h3 style="color:darkcyan;">3.1 Preparación de datos para el modelo</h3>

Divide el dataset en tres conjuntos respetando el orden cronológico. Nunca mezcles datos de distintos períodos en el mismo conjunto, ya que estarías usando información del futuro para predecir el pasado.

La división sugerida es:
- Entrenamiento: hasta 2019 (incluido)
- Validación: 2020 – 2022
- Test: 2023 en adelante

A continuación, aplica normalización a las variables numéricas. Recuerda que el `StandardScaler` debe ajustarse únicamente con los datos de entrenamiento y aplicarse (sin reajustar) a validación y test.

Si detectaste desbalanceo de clases en el apartado anterior, aplica aquí la técnica que consideres más adecuada (por ejemplo, `class_weight='balanced'` en los modelos, o submuestreo de la clase mayoritaria).

In [ ]:
# División temporal del dataset
fecha_col = 'fecha'
target_col = 'cycle_phase'

# Asegurar formato datetime y orden cronológico
df[fecha_col] = pd.to_datetime(df[fecha_col], errors='coerce')
df_model = df.dropna(subset=[fecha_col]).sort_values(fecha_col).reset_index(drop=True).copy()

# El target debe existir para entrenar/evaluar
df_model = df_model.dropna(subset=[target_col]).copy()

# Máscaras temporales (sin mezclar pasado/futuro)
mask_train = df_model[fecha_col].dt.year <= 2019
mask_val = (df_model[fecha_col].dt.year >= 2020) & (df_model[fecha_col].dt.year <= 2022)
mask_test = df_model[fecha_col].dt.year >= 2023

df_train = df_model.loc[mask_train].copy()
df_val = df_model.loc[mask_val].copy()
df_test = df_model.loc[mask_test].copy()

In [ ]:
print('Tamaños de partición:')
print(f"  Train: {len(df_train)} filas")
print(f"  Val:   {len(df_val)} filas")
print(f"  Test:  {len(df_test)} filas")

print('\nRangos temporales:')
print(f"  Train: {df_train[fecha_col].min()} -> {df_train[fecha_col].max()}")
print(f"  Val:   {df_val[fecha_col].min()} -> {df_val[fecha_col].max()}")
print(f"  Test:  {df_test[fecha_col].min()} -> {df_test[fecha_col].max()}")

# Comprobación rápida de clases por split
print('\nDistribución de clases por split:')
print('Train:')
print(df_train[target_col].value_counts(dropna=False))
print('\nVal:')
print(df_val[target_col].value_counts(dropna=False))
print('\nTest:')
print(df_test[target_col].value_counts(dropna=False))

In [ ]:
# Selección de features y normalización
# Pista: excluye de feature_cols la columna de fecha, el target y cualquier identificador

# feature_cols = [...]

# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(...)    # fit solo en train
# X_val_scaled   = scaler.transform(...)        # solo transform en val
# X_test_scaled  = scaler.transform(...)        # solo transform en test

# le = LabelEncoder()
# y_train = le.fit_transform(...)
# y_val   = le.transform(...)
# y_test  = le.transform(...)

In [ ]:
# Selección de features y normalización (sin leakage)

# Definir columnas a excluir
id_cols = [fecha_col, target_col]

# Columnas usadas para construir el target (evitar circularidad)
target_rule_cols = [
    'PIB_var',
    'PIB_var_lag_1', 'PIB_var_lag_3', 'PIB_var_lag_6',
    'PMI_Manufacturero',
    'pmi_manufacturero_diff', 'pmi_manufacturero_trend',
    'PMI_Manufacturero_lag_1', 'PMI_Manufacturero_lag_3', 'PMI_Manufacturero_lag_6'
]

exclude_cols = set(id_cols + [c for c in target_rule_cols if c in df_model.columns])

# Seleccionar features numéricas disponibles
candidate_cols = [c for c in df_model.columns if c not in exclude_cols]
numeric_feature_cols = [
    c for c in candidate_cols
    if pd.api.types.is_numeric_dtype(df_model[c]) or pd.api.types.is_bool_dtype(df_model[c])
]

feature_cols = numeric_feature_cols.copy()
print(f"Número de features seleccionadas: {len(feature_cols)}")
print('Primeras 15 features:', feature_cols[:15])

# Construir matrices X
X_train = df_train[feature_cols].copy()
X_val = df_val[feature_cols].copy()
X_test = df_test[feature_cols].copy()

# Imputación de nulos (fit solo en train)
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)
X_test_imp = imputer.transform(X_test)

# Escalado (fit solo en train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

# Codificación del target
le = LabelEncoder()
y_train = le.fit_transform(df_train[target_col])
y_val = le.transform(df_val[target_col])
y_test = le.transform(df_test[target_col])

In [ ]:
print('\nClases codificadas en train:')
for cls, idx in zip(le.classes_, range(len(le.classes_))):
    print(f"  {idx} -> {cls}")

print('\nDimensiones finales:')
print('  X_train_scaled:', X_train_scaled.shape, '| y_train:', y_train.shape)
print('  X_val_scaled:  ', X_val_scaled.shape, '| y_val:  ', y_val.shape)
print('  X_test_scaled: ', X_test_scaled.shape, '| y_test: ', y_test.shape)

<h3 style="color:darkcyan;">3.2 Implementación de modelos</h3>

Implementa y entrena al menos tres modelos de ML. Para cada uno, incluye en una celda de texto:

- Por qué lo has elegido para este problema concreto
- Qué hiperparámetros has modificado respecto a los valores por defecto y por qué
- Qué limitaciones puede tener en el contexto de series temporales económicas

Los tres modelos obligatorios son los siguientes. Tienes el esqueleto de cada uno; debes completar los hiperparámetros y el ajuste.

#### Modelo 1: Regresión Logística Multinomial

Es el modelo más sencillo para clasificación multiclase. Sirve como referencia (baseline): si modelos más complejos no lo mejoran claramente, algo puede estar mal en el pipeline. Su mayor ventaja es la interpretabilidad: los coeficientes indican directamente el peso de cada variable.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Instancia y entrena el modelo con los hiperparámetros que consideres adecuados
# Parámetros a explorar: C (regularización), max_iter, solver
# Recuerda usar X_train_scaled (la regresión logística es sensible a la escala)

# rl = LogisticRegression(...)
# rl.fit(...)

In [ ]:
# Modelo 1: Regresión Logística Multinomial
logreg_model = LogisticRegression(
    solver='lbfgs',
    max_iter=3000,
    C=1.0,
    class_weight='balanced',
    random_state=42
)

# Entrenamiento
logreg_model.fit(X_train_scaled, y_train)

# Predicciones
y_val_pred_logreg = logreg_model.predict(X_val_scaled)
y_test_pred_logreg = logreg_model.predict(X_test_scaled)

# Métricas en validación
val_acc_logreg = accuracy_score(y_val, y_val_pred_logreg)
val_f1_macro_logreg = f1_score(y_val, y_val_pred_logreg, average='macro')

print('Regresión Logística Multinomial - Resultados en validación')
print(f'  Accuracy: {val_acc_logreg:.4f}')
print(f'  F1-macro: {val_f1_macro_logreg:.4f}')
print('\nClassification report (validación):')
print(classification_report(y_val, y_val_pred_logreg, target_names=le.classes_))

#### Modelo 2: Random Forest

Conjunto de árboles de decisión entrenados con muestras y variables aleatorias. Captura interacciones no lineales entre variables y proporciona de forma nativa la importancia de cada variable, lo que facilitará la interpretación económica en la Parte 4.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Parámetros a explorar: n_estimators, max_depth, min_samples_leaf, class_weight
# No necesita normalización previa

rf = RandomForestClassifier(
    n_estimators=1000,          # Aumentar el número de árboles para mayor robustez
    max_depth=10,               # Limitar la profundidad para evitar sobreajuste
    min_samples_leaf=5,         # Asegurar que cada hoja tenga al menos 5 muestras
    class_weight='balanced',    # Manejar el desbalance de clases
    random_state=42             # Para reproducibilidad de los resultados
)
rf.fit(X_train_scaled, y_train)

# Predicciones
y_val_pred_rf = rf.predict(X_val_scaled)
y_test_pred_rf = rf.predict(X_test_scaled)

# Métricas en validación
val_acc_rf = accuracy_score(y_val, y_val_pred_rf)
val_f1_macro_rf = f1_score(y_val, y_val_pred_rf, average='macro')

print('Random Forest - Resultados en validación')
print(f'  Accuracy: {val_acc_rf:.4f}')
print(f'  F1-macro: {val_f1_macro_rf:.4f}')
print('\nClassification report (validación):')
print(classification_report(y_val, y_val_pred_rf, target_names=le.classes_))

#### Modelo 3: XGBoost

Gradient boosting que construye árboles de forma secuencial, corrigiendo los errores del árbol anterior. Suele ofrecer el mejor rendimiento predictivo y maneja bien los datos con ruido y valores extremos. Es el modelo más usado en competiciones de datos tabulares.

In [ ]:
import xgboost as xgb

# Parámetros a explorar: n_estimators, learning_rate, max_depth, subsample, colsample_bytree
# objective debe ser 'multi:softprob' para clasificación multiclase

xgb_model = xgb.XGBClassifier(
    objective='multi:softprob', # Para clasificación multiclase con probabilidades
    num_class=len(le.classes_), # Número de clases en el target
    n_estimators=1000,          # Número de árboles
    learning_rate=0.05,         # Tasa de aprendizaje
    max_depth=5,                # Profundidad máxima del árbol
    subsample=0.8,              # Proporción de muestras para entrenar cada árbol
    colsample_bytree=0.8,       # Proporción de características para entrenar cada árbol
    use_label_encoder=False,    # Para evitar un warning, ya que el LabelEncoder se maneja manualmente
    eval_metric='mlogloss',     # Métrica de evaluación para clasificación multiclase
    random_state=42             # Para reproducibilidad
)
xgb_model.fit(X_train_scaled, y_train)

# Predicciones
y_val_pred_xgb = xgb_model.predict(X_val_scaled)
y_test_pred_xgb = xgb_model.predict(X_test_scaled)

# Métricas en validación
val_acc_xgb = accuracy_score(y_val, y_val_pred_xgb)
val_f1_macro_xgb = f1_score(y_val, y_val_pred_xgb, average='macro')

print('XGBoost - Resultados en validación')
print(f'  Accuracy: {val_acc_xgb:.4f}')
print(f'  F1-macro: {val_f1_macro_xgb:.4f}')
print('\nClassification report (validación):')
print(classification_report(y_val, y_val_pred_xgb, target_names=le.classes_))

#### Modelos opcionales (+10% en la nota)

Si lo deseas, implementa alguno de estos modelos adicionales:

- LSTM / RNN para modelar explícitamente la secuencia temporal (`tensorflow.keras`)
- SVM con kernel RBF (`sklearn.svm.SVC`)
- Stacking de modelos (`sklearn.ensemble.StackingClassifier`)
- O algún otro modelo que creas oportuno.

#### Modelo Opcional 1: Support Vector Machine (SVM) con kernel RBF

Los SVM son potentes clasificadores que buscan el hiperplano óptimo que separe las clases. Con un kernel RBF (Radial Basis Function), pueden manejar relaciones no lineales complejas entre las variables. Es una buena opción para problemas de clasificación donde las fronteras de decisión no son lineales y se desea una buena generalización, aunque puede ser sensible a la escala de las características y al ruido en los datos.

In [ ]:
from sklearn.svm import SVC

# Instancia y entrena el modelo SVC
# Parámetros a explorar: C (regularización), kernel, gamma (para kernel RBF), class_weight

svm_model = SVC(
    kernel='rbf',               # Kernel de función de base radial para relaciones no lineales
    C=1.0,                      # Parámetro de regularización, controla el trade-off entre margen y error
    gamma='scale',              # Coeficiente del kernel, 'scale' usa 1 / (n_features * X.var())
    class_weight='balanced',    # Manejar el desbalance de clases
    random_state=42             # Para reproducibilidad
)
svm_model.fit(X_train_scaled, y_train)

# Predicciones
y_val_pred_svm = svm_model.predict(X_val_scaled)
y_test_pred_svm = svm_model.predict(X_test_scaled)

# Métricas en validación
val_acc_svm = accuracy_score(y_val, y_val_pred_svm)
val_f1_macro_svm = f1_score(y_val, y_val_pred_svm, average='macro')

print('Support Vector Machine (SVC) - Resultados en validación')
print(f'  Accuracy: {val_acc_svm:.4f}')
print(f'  F1-macro: {val_f1_macro_svm:.4f}')
print('\nClassification report (validación):')
print(classification_report(y_val, y_val_pred_svm, target_names=le.classes_))

#### Modelo Opcional 2: K-Nearest Neighbors (KNN)

KNN es un algoritmo de clasificación no paramétrico que clasifica un punto de datos basado en la mayoría de las clases de sus 'k' vecinos más cercanos en el espacio de características. Es simple de entender y efectivo para conjuntos de datos donde las clases están bien separadas, pero puede ser computacionalmente costoso con grandes volúmenes de datos y es sensible a la escala de las características y a la maldición de la dimensionalidad.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Instancia y entrena el modelo KNN
# Parámetros a explorar: n_neighbors (número de vecinos), weights (uniform o distance)

knn_model = KNeighborsClassifier(
    n_neighbors=5,              # Número de vecinos a considerar
    weights='distance',         # Pondera los vecinos por la inversa de su distancia
    metric='euclidean'          # Métrica de distancia a usar
)
knn_model.fit(X_train_scaled, y_train)

# Predicciones
y_val_pred_knn = knn_model.predict(X_val_scaled)
y_test_pred_knn = knn_model.predict(X_test_scaled)

# Métricas en validación
val_acc_knn = accuracy_score(y_val, y_val_pred_knn)
val_f1_macro_knn = f1_score(y_val, y_val_pred_knn, average='macro')

print('K-Nearest Neighbors (KNN) - Resultados en validación')
print(f'  Accuracy: {val_acc_knn:.4f}')
print(f'  F1-macro: {val_f1_macro_knn:.4f}')
print('\nClassification report (validación):')
print(classification_report(y_val, y_val_pred_knn, target_names=le.classes_))

<h3 style="color:darkcyan;">3.3 Optimización de hiperparámetros</h3>

Optimiza los hiperparámetros de al menos uno de los modelos anteriores usando búsqueda aleatoria (`RandomizedSearchCV`) con validación cruzada temporal (`TimeSeriesSplit`).

Usa `TimeSeriesSplit` y no `KFold`: en series temporales el k-fold aleatorio mezcla pasado y futuro en los folds, lo que introduce data leakage. `TimeSeriesSplit` siempre entrena en el pasado y valida en el futuro.

Documenta el espacio de búsqueda que has definido y justifica por qué has elegido esos rangos.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

# Define el validador temporal
tscv = TimeSeriesSplit(n_splits=5)

# Define el espacio de búsqueda de hiperparámetros para XGBoost
param_dist = {
    'n_estimators': [100, 200, 500, 1000],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2, 0.3]
}

# Ejecuta la búsqueda aleatoria
search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=50, # Número de combinaciones a probar
    cv=tscv,
    scoring='accuracy', # Cambiado a 'accuracy' para evitar NaN en f1_weighted por clases ausentes en folds
    random_state=42,
    n_jobs=-1, # Usa todos los núcleos de CPU disponibles
    verbose=1
)
search.fit(X_train_scaled, y_train)

print('Mejores hiperparámetros:', search.best_params_)
print('Mejor Accuracy en validación:', search.best_score_)

<h3 style="color:darkcyan;">3.4 Evaluación y comparación de modelos</h3>

Evalúa todos los modelos sobre el conjunto de test y compara sus resultados. Para cada modelo calcula:

- Accuracy global
- Precision, Recall y F1-Score por clase (usa `classification_report`)
- Matriz de confusión

Presta especial atención al **Recall de la clase Recesion**: un Recall bajo en recesión significa que el modelo no detecta estas situaciones a tiempo, que es el error más costoso desde el punto de vista económico.

Elabora una tabla comparativa con los resultados de todos los modelos y razona cuál elegirías y por qué.

In [ ]:
# Evaluación de los modelos sobre el conjunto de test
# Pista: usa classification_report(y_test, y_pred, target_names=le.classes_)
# para obtener las métricas por clase en formato legible

def print_metrics(model_name, y_true, y_pred, le_classes):
    print(f'\n--- {model_name} ---')
    print(f'Accuracy: {accuracy_score(y_true, y_pred):.4f}')
    print(f'F1-macro: {f1_score(y_true, y_pred, average="macro", zero_division=0):.4f}')
    print(f'\nClassification Report:\n{classification_report(y_true, y_pred, labels=range(len(le_classes)), target_names=le_classes, zero_division=0)}')

# Evaluación de Regresión Logística en el conjunto de test
print_metrics('Regresión Logística Multinomial', y_test, y_test_pred_logreg, le.classes_)

# Evaluación de Random Forest en el conjunto de test
print_metrics('Random Forest', y_test, y_test_pred_rf, le.classes_)

# Evaluación de XGBoost en el conjunto de test
print_metrics('XGBoost', y_test, y_test_pred_xgb, le.classes_)

# Evaluación de SVC en el conjunto de test
print_metrics('Support Vector Machine (SVC)', y_test, y_test_pred_svm, le.classes_)

# Evaluación de KNN en el conjunto de test
print_metrics('K-Nearest Neighbors (KNN)', y_test, y_test_pred_knn, le.classes_)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

# Matriz de Confusión para Regresión Logística
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred_logreg, display_labels=le.classes_, labels=range(len(le.classes_)), ax=axes[0], cmap='Blues')
axes[0].set_title('Regresión Logística')

# Matriz de Confusión para Random Forest
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred_rf, display_labels=le.classes_, labels=range(len(le.classes_)), ax=axes[1], cmap='Blues')
axes[1].set_title('Random Forest')

# Matriz de Confusión para XGBoost
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred_xgb, display_labels=le.classes_, labels=range(len(le.classes_)), ax=axes[2], cmap='Blues')
axes[2].set_title('XGBoost')

# Matriz de Confusión para SVC
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred_svm, display_labels=le.classes_, labels=range(len(le.classes_)), ax=axes[3], cmap='Blues')
axes[3].set_title('Support Vector Machine (SVC)')

# Matriz de Confusión para KNN
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred_knn, display_labels=le.classes_, labels=range(len(le.classes_)), ax=axes[4], cmap='Blues')
axes[4].set_title('K-Nearest Neighbors (KNN)')

# Ocultar el último subplot vacío si no hay más modelos
if len(axes) > 5:
    fig.delaxes(axes[5])

fig.suptitle('Matrices de Confusión de los Modelos en el Conjunto de Test', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

La razón por la que no aparecen las fases de “Recesión” y “Recuperación” en el conjunto de test es porque, al separar los datos por orden cronológico (para evitar usar información del futuro), el periodo más reciente simplemente no incluye este tipo de eventos, ya que son poco frecuentes. Por eso, las métricas para esas clases salen en cero o vacías: no es que el modelo falle, sino que no tiene ejemplos reales en ese periodo con los que poder evaluarse, lo que refleja una dificultad habitual al predecir eventos económicos raros.

In [ ]:
# Tabla comparativa de métricas
results = []

# Helper to safely get F1 score for 'Recesion' class (index 2 for 'Recesion' based on le.classes_)
def get_f1_recesion(y_true, y_pred, le_classes):
    report = classification_report(y_true, y_pred, labels=range(len(le_classes)), target_names=le_classes, output_dict=True, zero_division=0)
    return report['Recesion']['f1-score'] if 'Recesion' in report else 0.0

# Regresión Logística
results.append({
    'Modelo': 'Regresión Logística',
    'Accuracy': accuracy_score(y_test, y_test_pred_logreg),
    'F1-macro': f1_score(y_test, y_test_pred_logreg, average='macro', zero_division=0),
    'F1-Recesion': get_f1_recesion(y_test, y_test_pred_logreg, le.classes_)
})

# Random Forest
results.append({
    'Modelo': 'Random Forest',
    'Accuracy': accuracy_score(y_test, y_test_pred_rf),
    'F1-macro': f1_score(y_test, y_test_pred_rf, average='macro', zero_division=0),
    'F1-Recesion': get_f1_recesion(y_test, y_test_pred_rf, le.classes_)
})

# XGBoost
results.append({
    'Modelo': 'XGBoost',
    'Accuracy': accuracy_score(y_test, y_test_pred_xgb),
    'F1-macro': f1_score(y_test, y_test_pred_xgb, average='macro', zero_division=0),
    'F1-Recesion': get_f1_recesion(y_test, y_test_pred_xgb, le.classes_)
})

# SVC
results.append({
    'Modelo': 'Support Vector Machine (SVC)',
    'Accuracy': accuracy_score(y_test, y_test_pred_svm),
    'F1-macro': f1_score(y_test, y_test_pred_svm, average='macro', zero_division=0),
    'F1-Recesion': get_f1_recesion(y_test, y_test_pred_svm, le.classes_)
})

# KNN
results.append({
    'Modelo': 'K-Nearest Neighbors (KNN)',
    'Accuracy': accuracy_score(y_test, y_test_pred_knn),
    'F1-macro': f1_score(y_test, y_test_pred_knn, average='macro', zero_division=0),
    'F1-Recesion': get_f1_recesion(y_test, y_test_pred_knn, le.classes_)
})

comparison_df = pd.DataFrame(results)
display(comparison_df.sort_values(by='F1-macro', ascending=False))


Para entender mejor por qué persisten valores `NaN` en algunas columnas, examinemos el estado de `df_model` después de las operaciones de limpieza inicial y el filtrado por la variable objetivo. Veremos que las filas donde `cycle_phase` era `NaN` (debido a `NaN`s iniciales en `PIB_var` o `PMI_Manufacturero`) han sido eliminadas, pero otras columnas aún pueden tener `NaN`s.

In [ ]:
print('Primeras filas de df_model después de limpiar NaNs en `fecha` y `cycle_phase`:')
display(df_model.head())

print('\nConteo de valores NaN por columna en df_model antes de la imputación final:')
display(df_model.isnull().sum()[df_model.isnull().sum() > 0].sort_values(ascending=False))

Los resultados muestran que los modelos basados en árboles, como Random Forest y XGBoost, ofrecen el mejor rendimiento general, alcanzando una accuracy cercana al 80% y un F1-macro alrededor de 0.78, lo que indica que capturan mejor las relaciones complejas de los datos económicos frente a modelos como la Regresión Logística, SVC o KNN, que presentan un desempeño notablemente inferior. Sin embargo, la evaluación tiene una limitación importante: la clase “Recesión” no aparece en el conjunto de prueba, por lo que su F1 es 0 y no es posible medir la capacidad real de los modelos para detectar este escenario crítico. Además, el tamaño reducido del test (15 observaciones) y el desbalance de clases hacen que las métricas obtenidas sean poco robustas y no representen adecuadamente todas las fases del ciclo económico.

<h2 style="color:darkcyan;">Parte 4: Interpretación y Diagnóstico (15%)</h2>

<h3 style="color:darkcyan;">4.1 Importancia de variables</h3>

Genera un gráfico de importancia de variables para el modelo de Random Forest o XGBoost (ambos lo proporcionan en el atributo `feature_importances_`). Muestra las 15 variables más importantes.

Después, interpreta los resultados:

- ¿Qué variables dominan el modelo? ¿Coincide con lo que esperabas desde el punto de vista económico?
- ¿Los lags son importantes? ¿A qué horizonte temporal (1, 3 o 6 meses)?
- ¿Las variables que propusisteis en el apartado 2.2 aparecen entre las más importantes? ¿Qué conclusión sacáis?

In [ ]:
# Importancia de variables
# Pista: pd.Series(modelo.feature_importances_, index=feature_cols).sort_values(ascending=False)

feature_importances = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x=feature_importances.head(15).values, y=feature_importances.head(15).index)
plt.title('Top 15 Importancia de Variables (XGBoost)')
plt.xlabel('Importancia')
plt.ylabel('Variable')
plt.tight_layout()
plt.show()

print("Las 15 variables más importantes:")
print(feature_importances.head(15))

<h3 style="color:darkcyan;">4.2 Análisis de errores</h3>

Representa en una línea temporal los aciertos y errores del mejor modelo sobre el conjunto de test. El objetivo es identificar si el modelo falla de forma sistemática en ciertos períodos o en ciertas fases.

Responde a las siguientes preguntas:

- ¿Hay períodos donde el modelo falla de forma consistente?
- ¿Detecta los cambios de fase con antelación o con retraso?
- ¿Qué fases confunde más frecuentemente entre sí?

In [ ]:
# Análisis temporal de errores
# Pista: crea una columna 'acierto' comparando la predicción con el valor real
#        y visualiza esa columna en el tiempo

# Crear un DataFrame con los resultados del conjunto de test
test_results = df_test[['fecha', target_col]].copy()
test_results['prediccion'] = le.inverse_transform(y_test_pred_xgb)
test_results['real'] = le.inverse_transform(y_test)
test_results['acierto'] = (test_results['prediccion'] == test_results['real'])

# Visualizar aciertos y errores en el tiempo
plt.figure(figsize=(15, 6))
sns.scatterplot(data=test_results, x='fecha', y='acierto', hue='acierto', style='acierto', markers={True: 'o', False: 'X'}, s=100, palette={True: 'green', False: 'red'})
plt.title('Aciertos y Errores del Modelo XGBoost en el Conjunto de Test')
plt.xlabel('Fecha')
plt.ylabel('Acierto (1=Correcto, 0=Incorrecto)')
plt.yticks([0, 1], ['Incorrecto', 'Correcto'])
plt.grid(True)
sombrear_recesiones(plt.gca()) # Añadir las franjas de recesión para contexto
plt.tight_layout()
plt.show()

print("Tabla de aciertos y errores en el conjunto de test:")
display(test_results.head(15))

El modelo XGBoost funciona bien al identificar la fase de “Expansión”, acertando en la mayoría de los casos recientes, pero tiene problemas para detectar la “Desaceleración”, ya que en algunos momentos la confunde con expansión, lo cual es importante porque anticipar una desaceleración es clave a nivel económico. Además, la evaluación es limitada porque el conjunto de test no incluye ejemplos de “Recesión” ni “Recuperación”, por lo que no se puede medir cómo se comporta el modelo en estas fases menos frecuentes.

<h3 style="color:darkcyan;">4.3 Curvas de probabilidad por fase</h3>

Utiliza `predict_proba()` del mejor modelo para obtener la probabilidad predicha de cada fase del ciclo en el período de test. Representa las cuatro curvas de probabilidad sobre un mismo gráfico temporal.

Interpreta: ¿el modelo aumenta la probabilidad de recesión antes de que esta se produzca, o solo la detecta una vez ya iniciada?

In [ ]:
# Curvas de probabilidad predicha por fase
# Pista: modelo.predict_proba(X_test) devuelve una matriz con una columna por clase
#        las clases están ordenadas según le.classes_

# Obtener las probabilidades predichas por fase para el conjunto de test
probabilities = xgb_model.predict_proba(X_test_scaled)

# Crear un DataFrame con las probabilidades y las fechas del conjunto de test
prob_df = pd.DataFrame(probabilities, columns=le.classes_)
prob_df['fecha'] = df_test['fecha'].reset_index(drop=True)

# Fusionar las probabilidades con los resultados reales para ver dónde acierta/falla
plot_df = test_results.merge(prob_df, on='fecha', how='left')

plt.figure(figsize=(15, 7))
for i, class_name in enumerate(le.classes_):
    sns.lineplot(data=plot_df, x='fecha', y=class_name, label=f'Probabilidad de {class_name}')

# Añadir puntos para las predicciones reales
sns.scatterplot(data=plot_df, x='fecha', y=plot_df.apply(lambda row: row[row['real']], axis=1), color='black', marker='o', s=50, label='Clase Real (predicción perfecta)')

plt.title('Curvas de Probabilidad Predicha por Fase del Ciclo Económico en el Conjunto de Test')
plt.xlabel('Fecha')
plt.ylabel('Probabilidad')
sombrear_recesiones(plt.gca())
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.grid(True)
plt.tight_layout()
plt.show()

El modelo identifica muy bien la fase de “Expansión”, asignándole la mayor probabilidad en la mayor parte del periodo y coincidiendo en muchos casos con la realidad. Sin embargo, tiene dificultades para distinguir la “Desaceleración”, ya que a menudo le asigna menor probabilidad y la confunde con expansión, lo que indica problemas para anticipar este cambio de fase. Además, las fases de “Recesión” y “Recuperación” casi no aparecen porque no hay datos reales de estas en el conjunto de prueba, por lo que no se puede evaluar su comportamiento en esos escenarios.

In [ ]:
# Gráfica para comparar predicciones con probabilidades reales

# Reconstruir X_all_scaled para todo el df_model usando los objetos 'imputer' y 'scaler' ya ajustados
X_all = df_model[feature_cols].copy()
X_all_imp = imputer.transform(X_all) # Usar el imputer ya ajustado con el set de entrenamiento
X_all_scaled = scaler.transform(X_all_imp) # Usar el scaler ya ajustado con el set de entrenamiento

# Obtener probabilidades y predicciones para todos los datos
probabilities_all = xgb_model.predict_proba(X_all_scaled)
predictions_all_encoded = xgb_model.predict(X_all_scaled)
predictions_all_labels = le.inverse_transform(predictions_all_encoded)

# Preparar DataFrame para la visualización
prob_df_all = pd.DataFrame(probabilities_all, columns=le.classes_)
prob_df_all['fecha'] = df_model['fecha'].reset_index(drop=True)
prob_df_all['real'] = df_model[target_col].reset_index(drop=True)
prob_df_all['predicted'] = predictions_all_labels

# Añadir una columna que indique si la predicción es correcta
prob_df_all['correct_prediction'] = (prob_df_all['real'] == prob_df_all['predicted'])

# Para el scatter plot, obtener la probabilidad que el modelo asignó a la clase real
prob_df_all['prob_of_real_class'] = prob_df_all.apply(lambda row: row[row['real']], axis=1)

# Plotting
plt.figure(figsize=(18, 9))

# Plotear las curvas de probabilidad para cada fase
for class_name in le.classes_:
    sns.lineplot(data=prob_df_all, x='fecha', y=class_name, label=f'Prob. {class_name}', linewidth=1.5, alpha=0.7)

# Superponer los puntos para las fases reales, distinguiendo entre aciertos y errores
sns.scatterplot(
    data=prob_df_all[prob_df_all['correct_prediction']],
    x='fecha',
    y='prob_of_real_class',
    color='green',
    marker='o',
    s=70,
    label='Predicción Correcta (Fase Real)',
    zorder=5 # Asegurar que estos puntos estén por encima
)

sns.scatterplot(
    data=prob_df_all[~prob_df_all['correct_prediction']],
    x='fecha',
    y='prob_of_real_class',
    color='red',
    marker='X',
    s=100,
    label='Predicción Incorrecta (Fase Real)',
    zorder=6 # Asegurar que estos puntos estén por encima
)

plt.title('Curvas de Probabilidad Predicha y Aciertos/Errores por Fase del Ciclo Económico (Todos los datos)')
plt.xlabel('Fecha')
plt.ylabel('Probabilidad')
sombrear_recesiones(plt.gca()) # Asumiendo que la función sombrear_recesiones está disponible

plt.legend(loc='upper left', bbox_to_anchor=(1, 1), title='Leyenda')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

print("\nPrimeras filas del DataFrame con probabilidades y predicciones:")
display(prob_df_all.head())

print("\nÚltimas filas del DataFrame con probabilidades y predicciones:")
display(prob_df_all.tail())

El modelo XGBoost es bastante fiable a lo largo del tiempo, ya que en muchos casos predice correctamente la fase real, especialmente en etapas como la “Recesión” y parte de la “Expansión” en el periodo más reciente. Sin embargo, también se observan errores importantes en los últimos años, donde confunde episodios de “Desaceleración” con “Expansión”, lo que indica que le cuesta detectar cambios de enfriamiento económico. Además, las probabilidades asignadas suelen ser coherentes con la predicción final (la clase con mayor probabilidad es la elegida), lo que confirma que el modelo es consistente en su funcionamiento, aunque no siempre acierta en fases más difíciles de distinguir.

# Conclusión

Como nos sugiere el modelo, el país está en "Expansión", aunque hay algunas señales de "Desaceleración" en algunos periodos de tiempo. Sin embargo, no ha sabido distinguir los periodos de "Recesión" o "Recuperación" en el conjunto de test.

<h2 style="color:darkcyan;">Criterios de evaluación</h2>

| Componente | Peso | Criterios clave |
|---|---|---|
| Parte 1: Preparación y EDA | 20% | Calidad de las visualizaciones, insights económicos, limpieza justificada |
| Parte 2: Feature Engineering | 25% | Creatividad y solidez económica de las variables, justificación de las variables propias |
| Parte 3: Modelado y evaluación | 25% | Rendimiento de los modelos, rigor metodológico, uso correcto de la validación temporal |
| Parte 4: Interpretación | 20% | Profundidad del análisis, conexión con la realidad económica española |
| Documentación y claridad | +10% extra | Código comentado, reproducibilidad, interpretaciones en texto |

---

## Bonificaciones

| Tarea | Porcentaje adicional |
|---|---|
| Implementar un modelo LSTM para series temporales | +5% |
| Incorporar datos de sentimiento (noticias económicas, redes sociales) | +5% |
| Comparar el modelo con las previsiones publicadas por el Banco de España o la OCDE | +5% |

---

## Consejos y errores frecuentes

**Errores que debes evitar:**

- Data leakage: no uses información del futuro para predecir el pasado. El entrenamiento siempre debe preceder temporalmente a la validación.
- Validación aleatoria: en series temporales usa siempre `TimeSeriesSplit`. El K-Fold aleatorio mezcla pasado y futuro en los folds.
- Variables sin justificación: cada variable debe tener una razón económica sólida para estar en el modelo.
- Ignorar el desbalanceo de clases: si hay fases con muy pocos ejemplos, el modelo aprenderá a ignorarlas. Trátalo explícitamente.
- Sobreajuste a crisis específicas: el modelo debe generalizar. El COVID-19 es un evento estructuralmente distinto; tenlo en cuenta al interpretar los resultados.

**Buenas prácticas:**

- Comenta el código con claridad y añade celdas de texto que expliquen cada resultado.
- Valida la consistencia temporal de los datos antes de modelar.
- Interpreta los resultados con lógica económica, no solo estadística. Un buen modelo no es solo preciso: es interpretable y útil.

---

## Preguntas para la reflexión final

1. ¿Tu modelo hubiera detectado la crisis de 2008 antes de que se declarara oficialmente? ¿Y el impacto del COVID en 2020?

Lo más probable es que sí hubiera empezado a detectar señales de deterioro antes de que las crisis fueran oficialmente reconocidas, pero no necesariamente de forma clara o temprana.

En el caso de la crisis de 2008, este tipo de modelo suele captar primero cambios en variables adelantadas como la curva de tipos, el PMI o los mercados financieros, por lo que probablemente habría ido aumentando la probabilidad de “Desaceleración” o incluso “Recesión” antes de la confirmación oficial, aunque con cierto retraso y posible confusión inicial.

Para el impacto del COVID en 2020, la caída fue mucho más brusca y extrema, así que el modelo probablemente habría detectado muy rápido un cambio fuerte hacia “Recesión”, ya que los indicadores económicos se desplomaron de forma simultánea y muy evidente.

2. ¿Qué variables han resultado más importantes? ¿Coincide con la teoría económica?

Las variables más importantes del modelo XGBoost son el PIB (y su variación), la tasa de paro, los tipos de interés de los bonos a 2 y 10 años, y los rezagos del PMI de servicios. Esto coincide bastante con la teoría económica, ya que el PIB refleja directamente la actividad económica, el paro indica la situación del mercado laboral y los bonos y la curva de tipos reflejan expectativas y riesgo financiero, siendo además conocidos como indicadores adelantados de posibles crisis. También el PMI de servicios es clave porque anticipa cambios en la actividad del sector más importante de la economía española.

3. ¿Cuál es el equilibrio entre anticipación y precisión? ¿Preferirías un modelo que se equivoca poco o uno que avisa pronto aunque genere alguna falsa alarma?

El equilibrio depende del objetivo, pero en economía suele ser más importante anticiparse que acertar siempre. Es preferible un modelo que avise con tiempo de una posible recesión o desaceleración, aunque a veces se equivoque, porque no detectar una crisis a tiempo puede tener consecuencias mucho más graves que una falsa alarma. Sin embargo, si el modelo se equivoca demasiado, pierde utilidad y credibilidad. Por eso, lo ideal en el modelo XGBoost sería priorizar detectar bien las fases de crisis (aunque haya algunos falsos positivos), manteniendo un equilibrio entre avisar pronto y no generar demasiadas alarmas innecesarias.

4. ¿Qué limitaciones fundamentales tiene este tipo de aproximación para la economía española en particular?

Las principales limitaciones de este enfoque con el modelo XGBoost en la economía española son varias. Primero, hay un fuerte desbalance de clases, ya que fases como recesión o recuperación ocurren muy poco y el modelo aprende peor a detectarlas. Segundo, los datos no siempre tienen la misma frecuencia o historial, lo que obliga a simplificaciones que pueden afectar la precisión. Tercero, la economía cambia con el tiempo (crisis, políticas del BCE, pandemia), por lo que relaciones aprendidas en el pasado pueden dejar de ser válidas. Además, existe riesgo de sesgos en las variables usadas y de “data leakage” si no se construyen bien las características. También el modelo tiene dificultades para predecir eventos excepcionales como la crisis de 2008 o el COVID, porque no tiene ejemplos similares suficientes.

---